<a href="https://colab.research.google.com/github/heberdavi/mba-engsoft-tcc/blob/main/notebooks/02-visualizacao_resultados.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Célula 1: Configuração e Criação da Pasta de Exportação
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from google.colab import drive
import os

# 1. Montagem do Drive
drive.mount('/content/drive')

# 2. Definição de Caminhos
DB_PATH = '/content/drive/MyDrive/mba-engsof-tcc/versao_final/data/base-dados.db'
EXPORT_PATH = '/content/drive/MyDrive/mba-engsof-tcc/versao_final/graficos-tcc'

# Cria a pasta de exportação se ela não existir
if not os.path.exists(EXPORT_PATH):
    os.makedirs(EXPORT_PATH)
    print(f"📂 Pasta criada: {EXPORT_PATH}")

def get_connection():
    return sqlite3.connect(DB_PATH)

# Garante suporte a acentuação e visual limpo
sns.set_context("paper", font_scale=1.2)

# Configurações para qualidade ds imagens
sns.set_theme(style="whitegrid")
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.family'] = 'arial'
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

print("✅ Ambiente configurado para exportação de imagens JPG.")

In [ ]:
# Célula 2: Análise da eficiência semântica
import os

# 1. Obtenção dos dados via SQL utilizando a função global get_connection()
conn = get_connection()

query = """
SELECT
    t.antidoto_referencia AS "Eixo",
    COUNT(vt.verso_id) AS "Total_Versos",
    SUM(CASE WHEN vs.label = 'POS' THEN 1 ELSE 0 END) AS "Antidotos_Cura",
    AVG(vs.score_pos - vs.score_neg) AS "Polaridade_Media",
    ROUND(
        CAST(SUM(CASE WHEN vs.label = 'POS' THEN 1 ELSE 0 END) AS FLOAT) / COUNT(vt.verso_id) * 100, 2
    ) AS "Eficiencia_Cura"
FROM verso_topico vt
JOIN topico t ON vt.topico_id = t.id
JOIN verso_sentimento vs ON vt.verso_id = vs.verso_id
WHERE vt.topico_id < 3
GROUP BY t.id, t.antidoto_referencia;
"""

df = pd.read_sql_query(query, conn)
conn.close()

# 2. Mapeamento de Cores Conforme Premissas
color_map = {
    'Vazio vs. Propósito': 'green',
    'Exaustão vs. Refrigério': 'red',
    'Transitoriedade vs. Solidez': 'blue'
}
df['Cor'] = df['Eixo'].map(color_map)

# 3. Configuração do Gráfico
fig, ax = plt.subplots(figsize=(11, 7.5))

# Escalonamento das bolhas (baseado no volume total de versos)
# Reduzi levemente o fator de escala para evitar sobreposição excessiva
sizes = np.sqrt(df['Total_Versos']) * 70

# Geração das bolhas
scatter = ax.scatter(
    df['Polaridade_Media'],
    df['Antidotos_Cura'],
    s=sizes,
    c=df['Cor'],
    alpha=0.6,
    edgecolors="black",
    linewidth=1.2
)

# 4. Estilização dos Eixos (Rigor Acadêmico e Premissas)
ax.spines['left'].set_color('black')
ax.spines['left'].set_linewidth(1.5)
ax.spines['bottom'].set_color('black')
ax.spines['bottom'].set_linewidth(1.5)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(False)
ax.set_facecolor('white')

# Legendas dos eixos (Sem Negrito - Premissa)
ax.set_xlabel('Polaridade Média (Valência Emocional)', fontsize=12, fontweight='normal', labelpad=15)
ax.set_ylabel('Volume Absoluto de Antídotos (Cura)', fontsize=12, fontweight='normal', labelpad=15)

# 5. Rótulos e Legenda
for i, row in df.iterrows():
    # AJUSTE: Nome do Eixo + Quantidade de Antídotos (Mais próximo da bolha)
    # Offset reduzido de 0.05 para 0.03 para maior proximidade
    ax.text(
        row['Polaridade_Media'],
        row['Antidotos_Cura'] + (sizes[i] * 0.03),
        f"{row['Eixo']}\n({row['Antidotos_Cura']} Antídotos)",
        ha='center', va='bottom', fontsize=9.5, color='black',
        linespacing=1.2
    )

    # Valor de eficiência dentro da bolha
    ax.text(
        row['Polaridade_Media'],
        row['Antidotos_Cura'],
        f"{row['Eficiencia_Cura']}%",
        ha='center', va='center', fontsize=9, color='white',
        bbox=dict(facecolor='black', alpha=0.4, edgecolor='none', boxstyle='round,pad=0.3')
    )

# Legenda de Volume (Baseada nas cores das premissas)
for eixo, cor in color_map.items():
    if eixo in df['Eixo'].values:
        total = df[df['Eixo'] == eixo]['Total_Versos'].values[0]
        ax.scatter([], [], c=cor, alpha=0.6, s=150, label=f"{eixo} (N={total})")

ax.legend(title="Volume Total (Corpus)", loc='lower right', frameon=False, fontsize=10, title_fontsize=11)

# Linha de Referência (Neutralidade)
ax.axvline(0, color='black', linestyle=':', linewidth=1, alpha=0.2)

# Ajuste de Limites para evitar cortes
ax.set_xlim(df['Polaridade_Media'].min() - 0.08, df['Polaridade_Media'].max() + 0.08)
ax.set_ylim(-100, df['Antidotos_Cura'].max() + 350)

plt.tight_layout()

# 6. Salvamento no Google Drive
file_name = 'analise_eficiencia_semantica.jpg'
save_path = os.path.join(EXPORT_PATH, file_name)
plt.savefig(save_path, dpi=300, bbox_inches='tight')

print(f"📊 Gráfico exportado para: {save_path}")

In [ ]:
# Célula 3: Distribuição de ANTÍDOTOS (Positivos) por Gênero Literário
import os

def gerar_grafico_antidotos_por_genero():
    # 1. Conexão via função global e consulta SQL
    conn = get_connection()

    query = """
        SELECT
            g.nome as Genero,
            t.antidoto_referencia as Antidoto,
            COUNT(vt.verso_id) as Frequencia
        FROM verso_topico vt
        JOIN topico t ON vt.topico_id = t.id
        JOIN verso_sentimento vs ON vt.verso_id = vs.verso_id
        JOIN verso v ON vt.verso_id = v.id
        JOIN livro l ON v.livro_id = l.id
        JOIN genero_literario g ON l.genero_id = g.id
        WHERE vs.label = 'POS'         -- FILTRO: Apenas a Cura (Antídotos)
          AND t.id != 3                -- Exclui Narrativo/Normativo
        GROUP BY Genero, Antidoto
    """

    df_dist = pd.read_sql_query(query, conn)
    conn.close()

    if df_dist.empty:
        print("⚠️ Dados não encontrados para a geração do gráfico.")
        return

    # 2. Pivotar e Garantir Ordem Fixa de Cores e Nomenclatura
    df_pivot = df_dist.pivot(index='Genero', columns='Antidoto', values='Frequencia').fillna(0)

    # Ordem estrita conforme as premissas de cor
    ordem_eixos = [
        'Exaustão vs. Refrigério',   # RED
        'Vazio vs. Propósito',       # GREEN
        'Transitoriedade vs. Solidez' # BLUE
    ]

    # Reordenar colunas presentes
    colunas_presentes = [c for c in ordem_eixos if c in df_pivot.columns]
    df_pivot = df_pivot[colunas_presentes]

    # Ordenar Gêneros por volume total (efeito visual hierárquico)
    df_pivot['Total'] = df_pivot.sum(axis=1)
    df_pivot = df_pivot.sort_values(by='Total', ascending=True).drop(columns='Total')

    # 3. Configuração Estética (Premissas Acadêmicas)
    fig, ax = plt.subplots(figsize=(12, 8))
    fig.patch.set_facecolor('white')
    ax.set_facecolor('white')

    cores_premissas = ['red', 'green', 'blue']

    df_pivot.plot(kind='barh',
                  stacked=True,
                  color=cores_premissas,
                  ax=ax,
                  width=0.8,
                  edgecolor='white',
                  linewidth=0.5)

    # 4. Estilização dos Eixos (Premissas de 1.5pt e Cor Preta)
    ax.spines['left'].set_color('black')
    ax.spines['left'].set_linewidth(1.5)
    ax.spines['bottom'].set_color('black')
    ax.spines['bottom'].set_linewidth(1.5)

    # Remover bordas desnecessárias e grade
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(False)

    # Rótulos (Sem Negrito conforme premissa)
    ax.set_xlabel('Volume de Antídotos (Versículos com Sentimento Positivo)', fontsize=12, fontweight='normal', labelpad=15)
    ax.set_ylabel('Gênero Literário', fontsize=12, fontweight='normal', labelpad=15)

    # 5. Adicionando Data Labels (Números dentro das fatias)
    for p in ax.patches:
        width = p.get_width()
        if width > 10: # Só rotula fatias com volume significativo para não poluir
            ax.annotate(f'{int(width)}',
                        (p.get_x() + width / 2, p.get_y() + p.get_height() / 2),
                        ha='center', va='center',
                        fontsize=10, color='white', fontweight='bold')

    # 6. Legenda Externa (Sem Negrito)
    plt.legend(title='Eixos Existenciais',
               bbox_to_anchor=(1.0, 0.5),
               loc='center left',
               frameon=False,
               fontsize=10,
               title_fontsize=11)

    plt.tight_layout()

    # 7. Exportação para o Drive
    file_name = "analise_antidotos_por_genero.jpg"
    save_path = os.path.join(EXPORT_PATH, file_name)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')

    print(f"✅ Gráfico exportado com sucesso para: {save_path}")
    plt.show()

# Execução
gerar_grafico_antidotos_por_genero()

In [ ]:
# Célula 4: Nuvem de Palavras (WordCloud), Antídotos - Eixo Transitoriedade vs. Solidez, Gênero Literário Poético/Sapiencial
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from wordcloud import WordCloud
import sqlite3

# Supondo que as premissas de DB_PATH e EXPORT_PATH e get_connection() já estejam definidas

def gerar_wordcloud_solidez_poetica_refinada():
    # 1. Conexão e Consulta SQL: Busca os textos LEMATIZADOS (verso_limpo)
    conn = get_connection()

    query = """
      SELECT vl.texto_limpo
      FROM verso v
      JOIN livro l ON l.id = v.livro_id
      JOIN genero_literario gl ON gl.id = l.genero_id
      JOIN verso_topico vt ON vt.verso_id = v.id
      JOIN topico t ON t.id = vt.topico_id
      JOIN verso_sentimento vs ON vs.verso_id = v.id
      JOIN verso_limpo vl ON vl.verso_id = v.id
      WHERE gl.nome = 'Poético/Sapiencial'
      AND t.antidoto_referencia = 'Transitoriedade vs. Solidez'
      AND vs.label = 'POS'
    """

    df_textos = pd.read_sql_query(query, conn)
    conn.close()

    if df_textos.empty:
        print("⚠️ Nenhum texto lematizado encontrado para os critérios selecionados.")
        return

    # 2. Pré-processamento e Stopwords Customizadas (Agressivas)
    # Esta lista remove os ruídos gramaticais e teológicos comuns identificados na sua imagem anterior
    textos_unidos = " ".join(df_textos['texto_limpo'].values).lower()

    # Lista expandida para focar exclusivamente em antídotos
    stopwords_customizadas = {
        # Gramaticais e verbos comuns
        'ter', 'ser', 'vir', 'fazer', 'coisa', 'ir', 'estar', 'ver', 'dar', 'ficar',
        'todo', 'cada', 'outro', 'assim', 'apenas', 'então', 'ainda', 'mais', 'menos',
        'sempre', 'nunca', 'onde', 'quando', 'como', 'quem', 'qual', 'porque',
        'disso', 'disso', 'daquele', 'daquela', 'daquilo',

        # Teológicos gramaticais comuns (Se necessário filtrar)
        'senhor', 'deus', 'filho', 'pai', 'espírito', 'alguém', 'ninguém'
    }

    # Tons de azul para manter a identidade do eixo Transitoriedade vs. Solidez
    cores_premissa = 'Blues'

    # 3. Configuração da Nuvem de Palavras
    wc = WordCloud(
        width=1200,
        height=800,
        background_color='white',
        max_words=100,             # Mantemos as 100 mais frequentes
        stopwords=stopwords_customizadas, # APLICANDO A LIMPEZA AGRESSIVA
        colormap=cores_premissa,   # Azul conforme premissa
        prefer_horizontal=0.8,     # Maioria horizontal para legibilidade
        min_font_size=10,
        max_font_size=110,
        random_state=42            # Para reprodutibilidade
    ).generate(textos_unidos)

    # 4. Geração da Figura (Sem Bordas/Grades - Premissa Acadêmica)
    fig, ax = plt.subplots(figsize=(12, 8))
    ax.imshow(wc, interpolation='bilinear')

    # Estilização conforme premissas
    ax.axis('off') # Nuvens de palavras não possuem eixos
    fig.patch.set_facecolor('white')

    plt.tight_layout(pad=0)

    # 5. Exportação para o Drive no formato JPG
    file_name = "wordcloud_solidez_poetica_antidotos.jpg"
    save_path = os.path.join(EXPORT_PATH, file_name)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')

    print(f"✅ Nuvem de antídotos exportada para: {save_path}")
    plt.show()

# Execução da função
gerar_wordcloud_solidez_poetica_refinada()

In [ ]:
# Célula 5: Scatter Plot de Auditoria (XAI) - Eixo Vazio vs. Propósito, gênero Profético
import os
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

# A função get_connection() deve estar definida em uma célula anterior (Célula 1 de configuração)

def gerar_scatter_auditoria_completo():
    conn = get_connection() # Utiliza a função global da Célula 1

    query = """
      SELECT l.abreviacao||' '||v.numero_capitulo||':'||v.numero_verso AS Ref,
        v.texto, vs.score_pos AS Score, vt.gap_confianca AS Gap
      FROM genero_literario gl
      JOIN livro l ON l.genero_id = gl.id
      JOIN verso v ON v.livro_id = l.id
      JOIN verso_topico vt ON vt.verso_id = v.id
      JOIN topico t ON t.id = vt.topico_id
      JOIN verso_sentimento vs ON vs.verso_id = v.id
      WHERE gl.nome = 'Profético'
      AND t.antidoto_referencia = 'Vazio vs. Propósito'
      AND vs.label = 'POS'
    """

    df = pd.read_sql_query(query, conn)
    conn.close()

    if df.empty:
        print("⚠️ Nenhum dado encontrado para os filtros selecionados.")
        return

    # 1. Configuração da Figura (Estilo Acadêmico Limpo)
    fig, ax = plt.subplots(figsize=(12, 9))
    fig.patch.set_facecolor('white')
    ax.set_facecolor('white')

    # 2. Geração do Scatter (Cor Verde conforme premissa de Vazio x Propósito)
    ax.scatter(df['Gap'], df['Score'],
               color='green', s=100, alpha=0.5,
               edgecolors='black', linewidth=0.8)

    # 3. Estilização dos Eixos (Premissa rigorosa: 1.5pt, Preto, Sem Negrito)
    ax.spines['left'].set_color('black')
    ax.spines['left'].set_linewidth(1.5)
    ax.spines['bottom'].set_color('black')
    ax.spines['bottom'].set_linewidth(1.5)

    # Remover bordas superiores/direitas e grade (Premissa de visual técnico)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(False)

    # Legendas dos Eixos (Fonte sem negrito conforme premissa)
    ax.set_xlabel('Gap de Confiança (Assertividade do Modelo - XAI)', fontsize=12, fontweight='normal', labelpad=12)
    ax.set_ylabel('Score de Positividade (Resolutividade)', fontsize=12, fontweight='normal', labelpad=12)

    # 4. Rotulagem Manual com Regra de Exceção para evitar sobreposição
    for i, row in df.iterrows():
        # Lógica padrão de deslocamento alternado para a maioria dos pontos
        offset_x = 5 if i % 2 == 0 else -35
        offset_y = 5 if i % 3 == 0 else -12

        # --- AJUSTE TÉCNICO ESPECÍFICO PARA Is 62:3 ---
        # Move o rótulo para a DIREITA do ponto para não sobrepor a descrição do quadrante (top-left)
        if row['Ref'] == 'Is 62:3':
            offset_x = 10
            offset_y = -8

        ax.annotate(row['Ref'],
                    (row['Gap'], row['Score']),
                    xytext=(offset_x, offset_y),
                    textcoords='offset points',
                    fontsize=8,
                    color='black',
                    alpha=0.9)

    # 5. Linhas de Quadrantes (Médias do conjunto)
    ax.axhline(df['Score'].mean(), color='black', linestyle=':', linewidth=1, alpha=0.3)
    ax.axvline(df['Gap'].mean(), color='black', linestyle=':', linewidth=1, alpha=0.3)

    # Rótulo do Quadrante de Ouro (Ancorado no canto superior esquerdo, em verde)
    ax.text(df['Gap'].min(), df['Score'].max(), 'Quadrante de Ouro\n(Alta Resolutividade e Certeza)',
            fontsize=9, color='green', verticalalignment='top', fontweight='bold', alpha=0.8)

    plt.tight_layout()

    # 6. Exportação para o Google Drive conforme EXPORT_PATH definido na Célula 1
    file_name = "scatter_auditoria_proposito_profetico.jpg"
    save_path = os.path.join(EXPORT_PATH, file_name)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')

    print(f"✅ Gráfico final com rótulos organizados exportado para: {save_path}")
    plt.show()

# Execução da função ajustada
gerar_scatter_auditoria_completo()

In [ ]:
# Célula 6: Distribuição de Resgate, por Gênero e Eixo
import os
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

def gerar_grafico_resgate():
    # 1. Extração via SQL
    conn = sqlite3.connect(DB_PATH)
    query = """
        SELECT gl.nome as genero, t.antidoto_referencia as eixo, count(1) as qt_ocor
        FROM verso v
        JOIN livro l ON l.id = v.livro_id
        JOIN genero_literario gl ON gl.id = l.genero_id
        JOIN verso_topico vt ON vt.verso_id = v.id
        JOIN topico t ON t.id = vt.topico_id
        JOIN verso_sentimento vs ON vs.verso_id = v.id
        WHERE vt.similaridade_final < vt.p_narrativo
          AND vs.label = 'POS'
        GROUP BY genero, eixo
    """
    df = pd.read_sql_query(query, conn)
    conn.close()

    if df.empty:
        print("⚠️ Dados não encontrados para os critérios de resgate.")
        return

    # 2. Mapeamento de Cores Estrito por Eixo
    color_map = {
        'Vazio vs. Propósito': '#2a9d8f',       # Verde
        'Exaustão vs. Refrigério': '#e63946',   # Vermelho
        'Transitoriedade vs. Solidez': '#457b9d' # Azul
    }

    df_pivot = df.pivot(index='genero', columns='eixo', values='qt_ocor').fillna(0)

    # Ordenação e Seleção de Colunas
    available_cols = [c for c in color_map.keys() if c in df_pivot.columns]
    colors = [color_map[c] for c in available_cols]

    df_pivot['total'] = df_pivot.sum(axis=1)
    df_pivot = df_pivot.sort_values(by='total', ascending=True)

    # 3. Execução do Plot
    ax = df_pivot[available_cols].plot(
        kind='barh',
        stacked=True,
        figsize=(12, 8),
        color=colors,
        width=0.75
    )

    # 4. Estilização Acadêmica (1.5pt, Sem Grade, Sem Título)
    plt.xlabel('Quantidade de Versículos (Resgate de Limite)', fontsize=12)
    plt.ylabel('Gênero Literário', fontsize=12)
    plt.legend(title="Eixo Existencial", bbox_to_anchor=(1.05, 1), loc='upper left')

    # Configuração da espessura dos eixos
    ax.spines['left'].set_color('black')
    ax.spines['left'].set_linewidth(1.5)
    ax.spines['bottom'].set_color('black')
    ax.spines['bottom'].set_linewidth(1.5)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.grid(False)

    # Rótulos internos
    for p in ax.patches:
        width = p.get_width()
        if width > 0:
            ax.annotate(f'{int(width)}',
                        (p.get_x() + width / 2, p.get_y() + p.get_height() / 2),
                        ha='center', va='center', color='white', fontsize=10, fontweight='bold')

    plt.tight_layout()

    # 5. Exportação
    save_path = os.path.join(EXPORT_PATH, "resgate_antidotos_genero_eixo.jpg")
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"✅ Gráfico exportado para: {save_path}")
    plt.show()

gerar_grafico_resgate()

In [ ]:
# Célula 7: Gráfico de amplitude emocional por gênero literário
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

def gerar_grafico_amplitude():
    # 1. Conexão e Extração (Utilizando sua consulta validada)
    conn = get_connection()
    query = """
    SELECT
        gl.nome AS genero,
        ROUND(MAX(vs.score_pos), 4) AS maior_score_positivo,
        ROUND(MAX(vs.score_neg), 4) AS maior_score_negativo,
        ROUND(AVG(CASE
            WHEN vs.label = 'POS' THEN vs.score_pos
            WHEN vs.label = 'NEG' THEN -vs.score_neg
            ELSE 0
        END), 4) AS polaridade_media
    FROM genero_literario gl
    JOIN livro l ON l.genero_id = gl.id
    JOIN verso v ON v.livro_id = l.id
    JOIN verso_sentimento vs ON vs.verso_id = v.id
    GROUP BY gl.nome;
    """
    df_resumo = pd.read_sql_query(query, conn)
    conn.close()

    if df_resumo.empty:
        print("⚠️ Sem dados. Verifique a tabela 'verso_sentimento'.")
        return

    # 2. Preparação dos Dados para o Plot
    # Ordenamos para garantir a estética "escadinha" (mais negativo embaixo)
    df_resumo = df_resumo.sort_values(by='polaridade_media', ascending=True).reset_index(drop=True)

    # 3. Configuração do Plot
    plt.figure(figsize=(14, 9))
    sns.set_style("white")

    for i, row in df_resumo.iterrows():
        # vmin é o limite da crise (negativo), vmax é o limite da cura (positivo)
        vmin = -row['maior_score_negativo']
        vmax = row['maior_score_positivo']
        vmedia = row['polaridade_media']

        # A. Linha de Amplitude (Hlines)
        plt.hlines(i, vmin, vmax, color='#D3D3D3', linewidth=2, zorder=0, alpha=0.7)

        # B. O Marcador da Média (Scatter)
        # Lógica de cor: Verde para suporte positivo, Coral para crise/negativo
        cor_ponto = '#2E8B57' if vmedia >= 0 else '#FF6B6B'
        plt.scatter(vmedia, i, color=cor_ponto, s=190, zorder=2, edgecolor='white', linewidth=1.2)

        # C. Anotações de Valores Reais (Extremos e Média)
        # Valor Negativo (Esquerda)
        plt.text(vmin - 0.03, i, f"{vmin:.4f}", va='center', ha='right', fontsize=9, color='#555')
        # Valor Positivo (Direita)
        plt.text(vmax + 0.03, i, f"{vmax:.4f}", va='center', ha='left', fontsize=9, color='#555')
        # Valor da Média (Acima do ponto)
        plt.text(vmedia, i + 0.28, f"{vmedia:.4f}", va='center', ha='center',
                 fontsize=8.5, fontweight='bold', color=cor_ponto)

    # 4. Refinamentos Estéticos
    plt.axvline(0, color='black', linestyle='--', alpha=0.2, linewidth=1, zorder=1)

    # Ajuste dos nomes dos gêneros no eixo Y
    plt.yticks(range(len(df_resumo)), df_resumo['genero'], fontsize=11)

    # Ajuste da escala do eixo X para cobrir de -1 a 1
    plt.xticks([-1, -0.75, -0.5, -0.25, 0, 0.25, 0.5, 0.75, 1], fontsize=10)
    plt.xlim(-1.25, 1.25)

    plt.xlabel('Intensidade Emocional: Extremos (Mín/Máx) e Média de Sentimento', fontsize=12, labelpad=15)
    plt.ylabel('Gêneros Literários Bíblicos', fontsize=12, labelpad=15)

    sns.despine(left=True, bottom=True)
    plt.tight_layout()

    # 5. Exportação e Exibição
    file_name = "amplitude_sentimento_generos_literarios.jpg"
    save_path = os.path.join(EXPORT_PATH, file_name)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

gerar_grafico_amplitude()

In [ ]:
# Célula 8: Geração de Gauges de Intensidade por Eixo Existencial
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import os
import seaborn as sns
import unicodedata

def slugify(text):
    """Normaliza o nome para o sistema de arquivos."""
    text = text.split('vs.')[0].strip().lower() # Pega apenas a primeira parte do eixo
    return "".join(c for c in unicodedata.normalize('NFKD', text)
                   if unicodedata.category(c) != 'Mn').replace(' ', '_')

def criar_gauge_intensidade(valor, nome_arquivo, cor_eixo):
    # Escala de Positividade (0.0 a 1.0)
    min_escala, max_escala = 0.0, 1.0

    # Cálculo da posição do ponteiro (0 a 180 graus)
    valor_clamped = max(min_escala, min(max_escala, valor))
    posicao_graus = ((valor_clamped - min_escala) / (max_escala - min_escala)) * 180

    fig, ax = plt.subplots(figsize=(7, 4))

    # 1. Arco de Fundo (Semicírculo cinza)
    ax.pie([180, 180], colors=['#F0F0F0', 'white'], startangle=180,
            wedgeprops={'width': 0.3, 'edgecolor': 'white', 'linewidth': 2})

    # 2. Ponteiro Indicador (Agulha na cor do eixo)
    # Criamos uma pequena fatia de 3 graus para simular a agulha
    ax.pie([posicao_graus - 1.5, 3, 180 - posicao_graus - 1.5, 180],
           colors=['none', cor_eixo, 'none', 'white'],
           startangle=180, counterclock=True,
           wedgeprops={'width': 0.35, 'edgecolor': 'none'})

    # 3. Valor Central
    # Exibição em formato decimal com 3 casas, conforme rigor acadêmico
    plt.text(0, 0.1, f"{valor:.3f}", ha='center', va='center',
             fontsize=48, fontweight='bold', color='#2d3436')

    # 4. Limpeza e Formatação (Premissas de 1.5pt)
    ax.axis('equal')
    ax.set_xticks([])
    ax.set_yticks([])
    sns.despine(left=True, bottom=True)

    # Exportação em Alta Resolução
    path_completo = os.path.join(EXPORT_PATH, nome_arquivo)
    plt.savefig(path_completo, dpi=300, bbox_inches='tight', pad_inches=0.1)
    plt.close(fig)
    print(f"✅ Gauge Gerado: {nome_arquivo} (Intensidade: {valor:.4f})")

def processar_gauges_positividade():
    conn = sqlite3.connect(DB_PATH)
    # Consulta SQL conforme sua solicitação
    query = """
        SELECT t.antidoto_referencia, AVG(vs.score_pos) as media_positividade
        FROM topico t
        JOIN verso_topico vt ON vt.topico_id = t.id
        JOIN verso_sentimento vs ON vs.verso_id = vt.verso_id
        WHERE t.id <> 3
          AND vs.label = 'POS'
        GROUP BY t.antidoto_referencia
        ORDER BY media_positividade DESC;
    """
    df = pd.read_sql_query(query, conn)
    conn.close()

    # Mapeamento de Cores Estrito por Eixo
    cores_eixos = {
        'Vazio vs. Propósito': '#2a9d8f',       # Verde
        'Exaustão vs. Refrigério': '#e63946',   # Vermelho
        'Transitoriedade vs. Solidez': '#457b9d' # Azul
    }

    for _, row in df.iterrows():
        eixo = row['antidoto_referencia']
        cor = cores_eixos.get(eixo, '#333')
        nome_arq = f"gauge_intensidade_{slugify(eixo)}.jpg"

        criar_gauge_intensidade(row['media_positividade'], nome_arq, cor)

# Execução
processar_gauges_positividade()

In [ ]:
# *******SEM USO ******* Célula 5: Comparativo Crise vs. Cura por Gênero Literário
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

def gerar_grafico_crise_vs_cura_robusto():
    conn = get_connection()

    # 1. Consulta SQL: Focada em garantir que todos os gêneros com dados apareçam
    # Usamos JOINs explícitos para garantir a integridade entre verso_id e verso_id
    query = """
        SELECT
            g.nome as Genero,
            SUM(CASE WHEN vs.sentimento_num = 1 THEN 1 ELSE 0 END) as Cura_Positivo,
            SUM(CASE WHEN vs.sentimento_num = -1 THEN 1 ELSE 0 END) as Crise_Negativo
        FROM genero_literario g
        JOIN livro l ON g.id = l.genero_id
        JOIN verso v ON l.id = v.livro_id
        JOIN verso_topico vt ON v.id = vt.verso_id
        JOIN verso_sentimento vs ON v.id = vs.verso_id
        WHERE vt.topico_id IN (0, 1, 2)
        GROUP BY g.nome
        HAVING (Cura_Positivo + Crise_Negativo) > 0
        ORDER BY (Cura_Positivo + Crise_Negativo) DESC
    """

    df_comp = pd.read_sql_query(query, conn)
    conn.close()

    if df_comp.empty:
        print("⚠️ A consulta não retornou dados. Verifique se as tabelas vt e vs estão povoadas.")
        return

    # 2. Preparação dos Dados (Melt)
    df_melted = df_comp.melt(id_vars='Genero', var_name='Tipo', value_name='Quantidade')
    df_melted['Tipo'] = df_melted['Tipo'].replace({
        'Cura_Positivo': 'Antídoto (Cura)',
        'Crise_Negativo': 'Problemática (Crise)'
    })

    # 3. Gráfico
    plt.figure(figsize=(14, 8))
    sns.set_style("whitegrid")

    # Cores acadêmicas: Verde para Cura, Coral para Crise
    paleta = {'Antídoto (Cura)': '#50C878', 'Problemática (Crise)': '#FF6B6B'}

    ax = sns.barplot(data=df_melted, x='Quantidade', y='Genero', hue='Tipo',
                     palette=paleta, edgecolor='white')

    # Ajustes estéticos
    plt.xlabel('Volume de Versículos (Frequência Absoluta)', fontsize=11)
    plt.ylabel('Gênero Literário', fontsize=11)
    plt.legend(title='Análise de Sentimento', loc='lower right', frameon=True)

    # Rótulos nas barras
    for p in ax.patches:
        val = int(p.get_width())
        if val > 0:
            ax.annotate(f'{val}',
                        (p.get_width(), p.get_y() + p.get_height() / 2),
                        ha='left', va='center', fontsize=9, xytext=(5, 0),
                        textcoords='offset points')

    sns.despine()
    plt.tight_layout()

    # 4. Salvamento
    file_name = "comparativo_crise_cura_final.jpg"
    plt.savefig(os.path.join(EXPORT_PATH, file_name), dpi=300, bbox_inches='tight')
    plt.show()
    print(f"✅ Gráfico salvo com sucesso: {file_name}")

gerar_grafico_crise_vs_cura_robusto()

In [ ]:
# *******SEM USO ******* Célula 6: Mapa de Calor de Resolutividade Existencial
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

def gerar_heatmap_resolutividade():
    conn = get_connection()

    # SQL para calcular o Saldo Líquido (Pos - Neg) / Total por Gênero e Eixo
    query = """
        SELECT
            g.nome as Genero,
            t.antidoto_referencia as Autor,
            CAST(SUM(CASE WHEN vs.sentimento_num = 1 THEN 1 ELSE 0 END) -
                 SUM(CASE WHEN vs.sentimento_num = -1 THEN 1 ELSE 0 END) AS FLOAT) /
                 COUNT(vs.verso_id) as Saldo
        FROM verso_topico vt
        JOIN topico t ON vt.topico_id = t.id
        JOIN verso_sentimento vs ON vt.verso_id = vs.verso_id
        JOIN verso v ON vt.verso_id = v.id
        JOIN livro l ON v.livro_id = l.id
        JOIN genero_literario g ON l.genero_id = g.id
        WHERE t.id IN (0, 1, 2)
        GROUP BY Genero, Autor
        HAVING COUNT(vs.verso_id) > 10
    """
    df = pd.read_sql_query(query, conn)
    conn.close()

    # Pivotar para o formato de matriz
    df_pivot = df.pivot(index='Autor', columns='Genero', values='Saldo').fillna(0)

    # Gráfico
    plt.figure(figsize=(14, 6))
    sns.heatmap(df_pivot, annot=True, cmap="RdYlGn", center=0, fmt=".3f",
                linewidths=.5, cbar_kws={'label': 'Índice de Resolutividade'})

    plt.title('Mapa de Calor: Eficácia do Antídoto por Gênero Literário', fontsize=14, pad=20)
    plt.xlabel('Gênero Literário', fontsize=12)
    plt.ylabel('Eixo Existencial (Autor)', fontsize=12)

    file_name = "heatmap_resolutividade_final.jpg"
    plt.savefig(os.path.join(EXPORT_PATH, file_name), dpi=300, bbox_inches='tight')
    plt.show()
    print(f"✅ Heatmap de Resolutividade salvo: {file_name}")

gerar_heatmap_resolutividade()

In [ ]:
# *******SEM USO ******* Célula 7: Distribuição de Densidade Emocional (Violin Plot)
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

def gerar_violin_plot_existencial():
    conn = get_connection()

    query = """
        SELECT
            t.antidoto_referencia as Autor,
            vs.sentimento_num as Sentimento
        FROM verso_topico vt
        JOIN topico t ON vt.topico_id = t.id
        JOIN verso_sentimento vs ON vt.verso_id = vs.verso_id
        WHERE t.id IN (0, 1, 2)
    """
    df = pd.read_sql_query(query, conn)
    conn.close()

    plt.figure(figsize=(12, 7))
    sns.set_style("white")

    # Cores harmonizadas com o restante do trabalho
    paleta = {'Esgotamento (Han)': '#2E8B57', 'Insignificância (Frankl)': '#4682B4', 'Transitoriedade (Bauman)': '#D2691E'}

    # O Violin Plot mostra a densidade de versos em cada ponto da escala -1 a 1
    ax = sns.violinplot(data=df, x='Autor', y='Sentimento', hue='Autor',
                        palette=paleta, inner="quart", bw_adjust=.5, legend=False)

    plt.axhline(0, color='black', linestyle='--', alpha=0.3)
    plt.title('Densidade e Distribuição de Sentimentos por Eixo Existencial', fontsize=14, pad=20)
    plt.ylabel('Escala de Sentimento (-1: Crise | 0: Neutro | +1: Cura)', fontsize=11)
    plt.xlabel('Eixo Filosófico', fontsize=11)

    file_name = "densidade_emocional_violin.jpg"
    plt.savefig(os.path.join(EXPORT_PATH, file_name), dpi=300, bbox_inches='tight')
    plt.show()
    print(f"✅ Violin Plot salvo com sucesso: {file_name}")

gerar_violin_plot_existencial()